In [2]:
import io
from pathlib import Path

import polars as pl
from google.cloud import bigquery

PROJECT = "project-16e0643c-a78c-4dd8-89b"
DATASET = "nyc_taxi"
LOCATION = "asia-south1"          # or "US"; must stay the same for the dataset

ZONE_CSV = Path("data/taxi_zone_lookup.csv")
TRIPS_DIR = Path("data/yellow_taxi_trip")
ZONE_TABLE = f"{PROJECT}.{DATASET}.taxi_zone_lookup"
TRIPS_TABLE = f"{PROJECT}.{DATASET}.yellow_taxi_trips"

client = bigquery.Client(project=PROJECT, location=LOCATION)
client.create_dataset(bigquery.Dataset(f"{PROJECT}.{DATASET}"), exists_ok=True)

files = sorted(TRIPS_DIR.glob("*.parquet"))
print(len(files), "trip files")

11 trip files


In [3]:
client.query(f"""
CREATE TABLE IF NOT EXISTS `{ZONE_TABLE}` (
  location_id  INT64  OPTIONS(description='TLC Taxi Zone ID (1-265). Joins to pu_location_id / do_location_id'),
  borough      STRING OPTIONS(description='Manhattan, Brooklyn, Queens, Bronx, Staten Island, EWR, Unknown, N/A'),
  zone         STRING OPTIONS(description='Neighbourhood or zone name, e.g. JFK Airport'),
  service_zone STRING OPTIONS(description='Yellow Zone (Manhattan core), Boro Zone (green-cab area), Airports, EWR, N/A')
)
OPTIONS(description='NYC TLC taxi zone lookup');
""").result()

client.query(f"""
CREATE TABLE IF NOT EXISTS `{TRIPS_TABLE}` (
  vendor_id             INT64     OPTIONS(description='TPEP provider: 1=Creative Mobile Technologies, 2=Curb Mobility, 6=Myle, 7=Helix'),
  pickup_datetime       TIMESTAMP OPTIONS(description='Meter engaged (trip start). NYC local time stored as-is'),
  dropoff_datetime      TIMESTAMP OPTIONS(description='Meter disengaged (trip end). NYC local time stored as-is'),
  passenger_count       INT64     OPTIONS(description='Number of passengers, entered by the driver'),
  trip_distance         FLOAT64   OPTIONS(description='Trip distance in miles from the taximeter'),
  ratecode_id           INT64     OPTIONS(description='1=Standard, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negotiated, 6=Group ride, 99=Unknown'),
  store_and_fwd_flag    STRING    OPTIONS(description='Y=stored in vehicle before sending, N=sent immediately'),
  pu_location_id        INT64     OPTIONS(description='Pickup TLC Taxi Zone. Join to taxi_zone_lookup.location_id'),
  do_location_id        INT64     OPTIONS(description='Dropoff TLC Taxi Zone. Join to taxi_zone_lookup.location_id'),
  payment_type          INT64     OPTIONS(description='0=Flex Fare, 1=Credit card, 2=Cash, 3=No charge, 4=Dispute, 5=Unknown, 6=Voided'),
  fare_amount           FLOAT64   OPTIONS(description='Time-and-distance meter fare, USD'),
  extra                 FLOAT64   OPTIONS(description='Misc extras and surcharges (rush hour, overnight), USD'),
  mta_tax               FLOAT64   OPTIONS(description='MTA tax, USD'),
  tip_amount            FLOAT64   OPTIONS(description='Tip, USD. Credit-card tips only; cash tips not recorded'),
  tolls_amount          FLOAT64   OPTIONS(description='Total tolls, USD'),
  improvement_surcharge FLOAT64   OPTIONS(description='Improvement surcharge, USD'),
  total_amount          FLOAT64   OPTIONS(description='Total charged to passengers excluding cash tips, USD'),
  congestion_surcharge  FLOAT64   OPTIONS(description='NY State congestion surcharge, USD'),
  airport_fee           FLOAT64   OPTIONS(description='LaGuardia/JFK pickup fee, USD'),
  cbd_congestion_fee    FLOAT64   OPTIONS(description='MTA Congestion Relief Zone toll (from Jan 2025), USD')
)
PARTITION BY DATE(pickup_datetime)
CLUSTER BY pu_location_id, do_location_id
OPTIONS(description='NYC TLC yellow taxi trip records. One row per trip. Partitioned by pickup date');
""").result()

print("tables ready")

tables ready


In [4]:
def load_polars(df: pl.DataFrame, table_id: str, mode: str = "WRITE_APPEND"):
    buf = io.BytesIO()
    df.write_parquet(buf)
    buf.seek(0)
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=mode,
        schema=client.get_table(table_id).schema,   # enforce the table's schema
    )
    client.load_table_from_file(buf, table_id, job_config=job_config).result()

In [5]:
zones = (
    pl.read_csv(ZONE_CSV)
    .rename({"LocationID": "location_id", "Borough": "borough",
             "Zone": "zone", "service_zone": "service_zone"})
    .with_columns(pl.col("location_id").cast(pl.Int64))
)
load_polars(zones, ZONE_TABLE, mode="WRITE_TRUNCATE")
print(client.get_table(ZONE_TABLE).num_rows, "zones loaded")

265 zones loaded


In [6]:
# lowercased source column -> (table column, Polars dtype matching the BigQuery type)
SCHEMA = {
    "vendorid":              ("vendor_id",             pl.Int64),
    "tpep_pickup_datetime":  ("pickup_datetime",       pl.Datetime("us")),
    "tpep_dropoff_datetime": ("dropoff_datetime",      pl.Datetime("us")),
    "passenger_count":       ("passenger_count",       pl.Int64),
    "trip_distance":         ("trip_distance",         pl.Float64),
    "ratecodeid":            ("ratecode_id",           pl.Int64),
    "store_and_fwd_flag":    ("store_and_fwd_flag",    pl.String),
    "pulocationid":          ("pu_location_id",        pl.Int64),
    "dolocationid":          ("do_location_id",        pl.Int64),
    "payment_type":          ("payment_type",          pl.Int64),
    "fare_amount":           ("fare_amount",           pl.Float64),
    "extra":                 ("extra",                 pl.Float64),
    "mta_tax":               ("mta_tax",               pl.Float64),
    "tip_amount":            ("tip_amount",            pl.Float64),
    "tolls_amount":          ("tolls_amount",          pl.Float64),
    "improvement_surcharge": ("improvement_surcharge", pl.Float64),
    "total_amount":          ("total_amount",          pl.Float64),
    "congestion_surcharge":  ("congestion_surcharge",  pl.Float64),
    "airport_fee":           ("airport_fee",           pl.Float64),
    "cbd_congestion_fee":    ("cbd_congestion_fee",    pl.Float64),
}


def normalize(path: Path) -> pl.DataFrame:
    """Rename and cast columns to the table schema. No rows are dropped."""
    lf = pl.scan_parquet(path)
    lf = lf.rename({c: c.lower() for c in lf.collect_schema().names()})  # Airport_fee -> airport_fee
    present = set(lf.collect_schema().names())

    missing = [src for src in SCHEMA if src not in present]
    if missing:
        print(f"  {path.name}: missing {missing} -> NULL")

    return lf.select([
        (pl.col(src) if src in present else pl.lit(None)).cast(dtype, strict=False).alias(dst)
        for src, (dst, dtype) in SCHEMA.items()
    ]).collect()

In [7]:
client.query(f"TRUNCATE TABLE `{TRIPS_TABLE}`").result()   # makes reruns safe

total_raw = 0
for f in files:
    raw_rows = pl.scan_parquet(f).select(pl.len()).collect().item()
    df = normalize(f)
    load_polars(df, TRIPS_TABLE)
    total_raw += raw_rows
    print(f"{f.name}: raw {raw_rows:,} -> loaded {df.height:,}")
    del df

yellow_tripdata_2025-01.parquet: raw 3,475,226 -> loaded 3,475,226
yellow_tripdata_2025-02.parquet: raw 3,577,543 -> loaded 3,577,543
yellow_tripdata_2025-03.parquet: raw 4,145,257 -> loaded 4,145,257
yellow_tripdata_2025-04.parquet: raw 3,970,553 -> loaded 3,970,553
yellow_tripdata_2025-05.parquet: raw 4,591,845 -> loaded 4,591,845
yellow_tripdata_2025-06.parquet: raw 4,322,960 -> loaded 4,322,960
yellow_tripdata_2025-08.parquet: raw 3,574,091 -> loaded 3,574,091
yellow_tripdata_2025-09.parquet: raw 4,251,015 -> loaded 4,251,015
yellow_tripdata_2025-10.parquet: raw 4,428,699 -> loaded 4,428,699
yellow_tripdata_2025-11.parquet: raw 4,181,444 -> loaded 4,181,444
yellow_tripdata_2025-12.parquet: raw 4,305,006 -> loaded 4,305,006


In [8]:
SILVER_TABLE = f"{PROJECT}.{DATASET}.silver_yellow_taxi_trips"

client.query(f"""
CREATE TABLE IF NOT EXISTS `{SILVER_TABLE}` (
  vendor_id             INT64     OPTIONS(description='TPEP provider: 1=Creative Mobile Technologies, 2=Curb Mobility, 6=Myle, 7=Helix'),
  pickup_datetime       TIMESTAMP OPTIONS(description='Meter engaged (trip start). NYC local time stored as-is'),
  dropoff_datetime      TIMESTAMP OPTIONS(description='Meter disengaged (trip end). NYC local time stored as-is'),
  pickup_date           DATE      OPTIONS(description='Pickup date. Partition column - filter on this for cheaper queries'),
  pickup_month          STRING    OPTIONS(description='Pickup month name, e.g. January'),
  pickup_day_of_week    STRING    OPTIONS(description='Pickup day name, e.g. Monday'),
  passenger_count       INT64     OPTIONS(description='Number of passengers, entered by the driver'),
  trip_distance         FLOAT64   OPTIONS(description='Trip distance in miles'),
  ratecode_id           INT64     OPTIONS(description='1=Standard, 2=JFK, 3=Newark, 4=Nassau/Westchester, 5=Negotiated, 6=Group ride, 99=Unknown'),
  store_and_fwd_flag    STRING    OPTIONS(description='Y=stored in vehicle before sending, N=sent immediately'),
  pu_location_id        INT64     OPTIONS(description='Pickup TLC Taxi Zone ID'),
  pu_borough            STRING    OPTIONS(description='Pickup borough: Manhattan, Brooklyn, Queens, Bronx, Staten Island, EWR, Unknown, N/A'),
  pu_zone               STRING    OPTIONS(description='Pickup zone name, e.g. JFK Airport, Midtown Center'),
  pu_service_zone       STRING    OPTIONS(description='Pickup service zone: Yellow Zone, Boro Zone, Airports, EWR, N/A'),
  do_location_id        INT64     OPTIONS(description='Dropoff TLC Taxi Zone ID'),
  do_borough            STRING    OPTIONS(description='Dropoff borough: Manhattan, Brooklyn, Queens, Bronx, Staten Island, EWR, Unknown, N/A'),
  do_zone               STRING    OPTIONS(description='Dropoff zone name, e.g. JFK Airport, Midtown Center'),
  do_service_zone       STRING    OPTIONS(description='Dropoff service zone: Yellow Zone, Boro Zone, Airports, EWR, N/A'),
  payment_type          STRING    OPTIONS(description='Payment method: Flex Fare, Credit card, Cash, No charge, Dispute, Unknown, Voided'),
  fare_amount           FLOAT64   OPTIONS(description='Time-and-distance meter fare, USD'),
  extra                 FLOAT64   OPTIONS(description='Misc extras and surcharges, USD'),
  mta_tax               FLOAT64   OPTIONS(description='MTA tax, USD'),
  tip_amount            FLOAT64   OPTIONS(description='Tip, USD. Credit-card tips only'),
  tolls_amount          FLOAT64   OPTIONS(description='Total tolls, USD'),
  improvement_surcharge FLOAT64   OPTIONS(description='Improvement surcharge, USD'),
  total_amount          FLOAT64   OPTIONS(description='Total charged to passengers excluding cash tips, USD'),
  congestion_surcharge  FLOAT64   OPTIONS(description='NY State congestion surcharge, USD'),
  airport_fee           FLOAT64   OPTIONS(description='LaGuardia/JFK pickup fee, USD'),
  cbd_congestion_fee    FLOAT64   OPTIONS(description='MTA Congestion Relief Zone toll (from Jan 2025), USD')
)
PARTITION BY pickup_date
CLUSTER BY pu_borough, do_borough, payment_type
OPTIONS(description='Silver yellow taxi trips: pickup and dropoff zone details joined, payment type decoded, month/day-of-week added. All bronze rows kept');
""").result()

print("silver table ready")

silver table ready


In [9]:
client.query(f"TRUNCATE TABLE `{SILVER_TABLE}`").result()   # makes reruns safe

job = client.query(f"""
INSERT INTO `{SILVER_TABLE}` (
  vendor_id, pickup_datetime, dropoff_datetime, pickup_date, pickup_month, pickup_day_of_week,
  passenger_count, trip_distance, ratecode_id, store_and_fwd_flag,
  pu_location_id, pu_borough, pu_zone, pu_service_zone,
  do_location_id, do_borough, do_zone, do_service_zone,
  payment_type, fare_amount, extra, mta_tax, tip_amount, tolls_amount,
  improvement_surcharge, total_amount, congestion_surcharge, airport_fee, cbd_congestion_fee
)
SELECT
  t.vendor_id,
  t.pickup_datetime,
  t.dropoff_datetime,
  DATE(t.pickup_datetime),
  FORMAT_TIMESTAMP('%B', t.pickup_datetime),
  FORMAT_TIMESTAMP('%A', t.pickup_datetime),
  t.passenger_count,
  t.trip_distance,
  t.ratecode_id,
  t.store_and_fwd_flag,
  t.pu_location_id,
  pz.borough,
  pz.zone,
  pz.service_zone,
  t.do_location_id,
  dz.borough,
  dz.zone,
  dz.service_zone,
  CASE t.payment_type
    WHEN 0 THEN 'Flex Fare'
    WHEN 1 THEN 'Credit card'
    WHEN 2 THEN 'Cash'
    WHEN 3 THEN 'No charge'
    WHEN 4 THEN 'Dispute'
    WHEN 5 THEN 'Unknown'
    WHEN 6 THEN 'Voided'
    ELSE 'Unknown'
  END,
  t.fare_amount,
  t.extra,
  t.mta_tax,
  t.tip_amount,
  t.tolls_amount,
  t.improvement_surcharge,
  t.total_amount,
  t.congestion_surcharge,
  t.airport_fee,
  t.cbd_congestion_fee
FROM `{TRIPS_TABLE}` t
LEFT JOIN `{ZONE_TABLE}` pz ON t.pu_location_id = pz.location_id
LEFT JOIN `{ZONE_TABLE}` dz ON t.do_location_id = dz.location_id
""")
job.result()
print(f"inserted {job.num_dml_affected_rows:,} rows")

inserted 44,823,639 rows


In [10]:
bronze = client.query(f"SELECT COUNT(*) FROM `{TRIPS_TABLE}`").result().to_arrow().column(0)[0].as_py()
silver = client.query(f"SELECT COUNT(*) FROM `{SILVER_TABLE}`").result().to_arrow().column(0)[0].as_py()
print(f"bronze: {bronze:,} | silver: {silver:,} | match: {bronze == silver}")

pl.from_arrow(client.query(f"""
    SELECT pu_borough, pu_zone, do_borough, do_zone, COUNT(*) AS trips
    FROM `{SILVER_TABLE}`
    GROUP BY 1, 2, 3, 4
    ORDER BY trips DESC
    LIMIT 10
""").result().to_arrow())

d:\data-analyst-ai\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


bronze: 44,823,639 | silver: 44,823,639 | match: True


pu_borough,pu_zone,do_borough,do_zone,trips
str,str,str,str,i64
"""Manhattan""","""Upper East Side South""","""Manhattan""","""Upper East Side North""",283904
"""Manhattan""","""Upper East Side North""","""Manhattan""","""Upper East Side South""",244535
"""Manhattan""","""Upper East Side South""","""Manhattan""","""Upper East Side South""",201261
"""Manhattan""","""Upper East Side North""","""Manhattan""","""Upper East Side North""",186370
"""Manhattan""","""Midtown Center""","""Manhattan""","""Upper East Side South""",131993
"""Manhattan""","""Upper East Side South""","""Manhattan""","""Midtown Center""",123026
"""Manhattan""","""Midtown Center""","""Manhattan""","""Upper East Side North""",106562
"""Manhattan""","""Upper East Side South""","""Manhattan""","""Midtown East""",98948
"""Queens""","""JFK Airport""","""Queens""","""JFK Airport""",98848
